In [1]:
# !pip install pandas 
# !pip install SQLAlchemy
# !pip install strands-agents
# !pip install ollama

### 1️⃣ Generate Random Sales Dataset

In [2]:
import pandas as pd
import random
from datetime import datetime, timedelta

products = [
    "laptop", "phone", "tablet", "headphones",
    "monitor", "keyboard", "mouse"
]

regions = ["north", "south", "east", "west"]

rows = []

for i in range(500):

    row = {
        "order_id": i + 1,
        "product": random.choice(products),
        "region": random.choice(regions),
        "price": random.randint(100, 2000),
        "quantity": random.randint(1, 5),
        "order_date": datetime.now() - timedelta(days=random.randint(0, 90))
    }

    rows.append(row)

df = pd.DataFrame(rows)

df["revenue"] = df["price"] * df["quantity"]

### 2️⃣ Store CSV into SQLite using SQLAlchemy

In [3]:
from sqlalchemy import create_engine

# Create SQLite DB
engine = create_engine("sqlite:///sales.db")

# Store data
df.to_sql(
    "sales",
    engine,
    if_exists="replace",
    index=False
)

print("Data loaded into SQLite database")

Data loaded into SQLite database


### 3️⃣ SQL Execution Tool

In [4]:
from strands import Agent, tool

@tool
def run_sql(query: str):

    try:
        df = pd.read_sql(query, engine)

        return df.head(20).to_dict()

    except Exception as e:
        return str(e)

### 4️⃣ SQL Agent

This agent:

- generates SQL
- executes it
- fixes errors if needed
- returns summary

In [21]:
from strands.models.ollama import OllamaModel

model = OllamaModel(
    host="http://localhost:11434",  # Ollama server address
    model_id="ministral-3:3b"               # Specify which model to use
)

test_agent = Agent(model=model)

# Use the agent
test_agent("What is 2 + 3 = ?")

2 + 3 = **5**

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': '2 + 3 = **5**'}]}, metrics=EventLoopMetrics(cycle_count=1, tool_metrics={}, cycle_durations=[40.592188596725464], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='63c1a8bb-24a6-4636-9706-f49f02a80a63', usage={'inputTokens': 9, 'outputTokens': 562, 'totalTokens': 571})], usage={'inputTokens': 9, 'outputTokens': 562, 'totalTokens': 571})], traces=[<strands.telemetry.metrics.Trace object at 0x7c9d3b65e210>], accumulated_usage={'inputTokens': 9, 'outputTokens': 562, 'totalTokens': 571}, accumulated_metrics={'latencyMs': 40563.069252}), state={}, interrupts=None, structured_output=None)

In [6]:


schema = """
Table: sales

Columns:
order_id
product
region 
price
quantity
order_date
revenue
"""

system_prompt = f"""
You are an expert SQL analyst.

Your task is to:
1. Convert the user's question into a SQL query
2. Execute the SQL query using the run_sql tool
3. Return the results to the user in a clear, readable format

Database schema:
{schema}

Important:
- All text values in the database are in lowercase
- Always use the run_sql tool to execute your SQL queries
- After getting results, summarize them for the user in natural language
- If there's an error, explain what went wrong and try to fix it
"""

sql_agent = Agent(
    model=model,
    tools=[run_sql],
    system_prompt=system_prompt
)

In [13]:
# Test 1: Simple aggregation query
result = sql_agent("total sale of laptop")
print(result)

I'll help you find the total sales of laptops. Let me query the database to get this information.
Tool #1: run_sql
The total sales of laptops is **$170,870**.

This represents the sum of all revenue generated from laptop sales in the database.The total sales of laptops is **$170,870**.

This represents the sum of all revenue generated from laptop sales in the database.



### 5️⃣ Advanced Query Examples

Let's test the agent with various types of queries to showcase its capabilities:

In [14]:
# Test 2: Complex aggregation with grouping
result = sql_agent("What is the total revenue by region?")
print(result)

I'll query the database to get the total revenue broken down by region.
Tool #2: run_sql
Here's the total revenue by region, ranked from highest to lowest:

1. **East Region**: $445,226
2. **West Region**: $383,044  
3. **North Region**: $350,356
4. **South Region**: $341,647

The East region generates the highest revenue at over $445k, while the South region has the lowest revenue at approximately $342k. The difference between the highest and lowest performing regions is about $103k.Here's the total revenue by region, ranked from highest to lowest:

1. **East Region**: $445,226
2. **West Region**: $383,044  
3. **North Region**: $350,356
4. **South Region**: $341,647

The East region generates the highest revenue at over $445k, while the South region has the lowest revenue at approximately $342k. The difference between the highest and lowest performing regions is about $103k.



In [15]:
# Test 3: Date filtering
result = sql_agent("Show me sales from the last 30 days")
print(result)

I'll query the database to show sales from the last 30 days.
Tool #3: run_sql
Here are the sales from the last 30 days (20 orders total):

**Recent Sales Summary:**

**Most Recent Sales (March 15, 2026):**
- Order #218: Headphones (North) - $580 revenue
- Order #70: Headphones (South) - $3,666 revenue  
- Order #48: Laptop (East) - $4,232 revenue

**Top Revenue Orders:**
1. Order #93: Laptop (West) - **$9,885** (March 14)
2. Order #338: Tablet (North) - **$7,388** (March 13)
3. Order #48: Laptop (East) - **$4,232** (March 15)
4. Order #321: Headphones (South) - **$4,024** (March 14)
5. Order #429: Keyboard (North) - **$4,045** (March 14)

**Product Breakdown:**
- **Headphones**: 8 orders
- **Laptops**: 3 orders
- **Keyboards**: 2 orders
- **Monitors**: 3 orders
- **Tablets**: 2 orders
- **Phones**: 1 order

**Regional Distribution:**
- **East**: 4 orders
- **North**: 5 orders
- **South**: 6 orders
- **West**: 5 orders

The sales show consistent activity across all regions with a good m

In [16]:
# Test 4: Top N query
result = sql_agent("Which product has the highest total revenue?")
print(result)

I'll query the database to find which product has the highest total revenue.
Tool #4: run_sql
Here are the products ranked by total revenue:

1. **Tablet**: $246,172 (highest revenue)
2. **Headphones**: $239,120
3. **Monitor**: $231,268
4. **Mouse**: $230,124
5. **Phone**: $211,966
6. **Keyboard**: $190,753
7. **Laptop**: $170,870

**Tablets** have the highest total revenue at **$246,172**, followed closely by headphones at $239,120. It's interesting to note that while laptops might be expected to generate the most revenue due to their typically higher unit prices, tablets actually lead in total revenue, likely due to higher sales volume or premium tablet models being sold.Here are the products ranked by total revenue:

1. **Tablet**: $246,172 (highest revenue)
2. **Headphones**: $239,120
3. **Monitor**: $231,268
4. **Mouse**: $230,124
5. **Phone**: $211,966
6. **Keyboard**: $190,753
7. **Laptop**: $170,870

**Tablets** have the highest total revenue at **$246,172**, followed closely b

In [17]:
# Test 5: Multi-step analysis
result = sql_agent("Compare the average order value between north and south regions")
print(result)

I'll query the database to compare the average order value between the north and south regions.
Tool #5: run_sql
Here's the comparison of average order value between the North and South regions:

**South Region:**
- Average order value: **$3,050.42**
- Total orders: 112

**North Region:**
- Average order value: **$2,944.17**
- Total orders: 119

**Key Insights:**

1. **South region has a higher average order value** by $106.25 ($3,050.42 vs $2,944.17)
2. **North region has more total orders** (119 vs 112), but lower average value per order
3. The difference represents about **3.6% higher** average order value in the South
4. Despite fewer orders, the South region likely generates similar or higher total revenue due to the higher average order value

This suggests that customers in the South region tend to purchase higher-value items or larger quantities per order compared to the North region.Here's the comparison of average order value between the North and South regions:

**South Regi

### 6️⃣ Error Handling & Recovery

Let's see how the agent handles errors and recovers:

In [18]:
# Test 6: Intentional error - wrong column name
result = sql_agent("What is the total sales for product_name = 'laptop'?")
print(result)

I'll query the database to get the total sales for laptops.
Tool #6: run_sql
The total sales for laptops is **$170,870**.

This represents the sum of all revenue generated from laptop sales across all regions in the database.The total sales for laptops is **$170,870**.

This represents the sum of all revenue generated from laptop sales across all regions in the database.



### 7️⃣ Multi-Turn Conversation

The agent maintains context across multiple queries:

In [19]:
# First query
result1 = sql_agent("What are the top 3 products by revenue?")
print("=== First Query ===")
print(result1)
print("\n")

# Follow-up query (uses context from previous)
result2 = sql_agent("What about the bottom 3?")
print("=== Follow-up Query ===")
print(result2)

I'll query the database to find the top 3 products by total revenue.
Tool #7: run_sql
Here are the **top 3 products by revenue**:

1. **Tablet**: $246,172
2. **Headphones**: $239,120  
3. **Monitor**: $231,268

**Key Insights:**
- Tablets lead with the highest revenue at over $246k
- The top 3 products are quite close in revenue, with only about $15k difference between tablets and monitors
- These three products together generate $716,560 in total revenue
- Headphones and monitors are nearly tied for second and third place, differing by less than $8k=== First Query ===
Here are the **top 3 products by revenue**:

1. **Tablet**: $246,172
2. **Headphones**: $239,120  
3. **Monitor**: $231,268

**Key Insights:**
- Tablets lead with the highest revenue at over $246k
- The top 3 products are quite close in revenue, with only about $15k difference between tablets and monitors
- These three products together generate $716,560 in total revenue
- Headphones and monitors are nearly tied for seco

In [20]:
sql_agent.messages

[{'role': 'user', 'content': [{'text': 'total sale of laptop'}]},
 {'role': 'assistant',
  'content': [{'text': "I'll help you find the total sales of laptops. Let me query the database to get this information."},
   {'toolUse': {'toolUseId': 'tooluse_NusCILiuFQsQ181NrKeBjz',
     'name': 'run_sql',
     'input': {'query': "SELECT SUM(revenue) as total_laptop_sales FROM sales WHERE product = 'laptop'"}}}]},
 {'role': 'user',
  'content': [{'toolResult': {'toolUseId': 'tooluse_NusCILiuFQsQ181NrKeBjz',
     'status': 'success',
     'content': [{'text': '{"total_laptop_sales": {"0": 170870}}'}]}}]},
 {'role': 'assistant',
  'content': [{'text': 'The total sales of laptops is **$170,870**.\n\nThis represents the sum of all revenue generated from laptop sales in the database.'}]},
 {'role': 'user',
  'content': [{'text': 'What is the total revenue by region?'}]},
 {'role': 'assistant',
  'content': [{'text': "I'll query the database to get the total revenue broken down by region."},
   {'t